## Midterm
Building a model to predict salary potential.

In [1]:
import pandas as pd
import numpy as np
import xgboost as xgb
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction import DictVectorizer
from sklearn.linear_model import Ridge
from sklearn.metrics import root_mean_squared_error
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
import requests

## Dataset
This midterm assignment is using the [Salary vs Experience](https://www.kaggle.com/datasets/mubeenshehzadi/salary-dataset) dataset (saved locally as `Salary_Data.csv`)

In [2]:
sal_df = pd.read_csv('Salary_Data.csv')
sal_df.columns = sal_df.columns.str.lower().str.replace(' ', '_')
sal_df.head()

,age,gender,education_level,job_title,years_of_experience,salary
0,32.0,Male,Bachelor's,Software Engineer,5.0,90000.0
1,28.0,Female,Master's,Data Analyst,3.0,65000.0
2,45.0,Male,PhD,Senior Manager,15.0,150000.0
3,36.0,Female,Bachelor's,Sales Associate,7.0,60000.0
4,52.0,Male,Master's,Director,20.0,200000.0


In [3]:
sal_df.isna().sum()

age                    2
gender                 2
education_level        3
job_title              2
years_of_experience    3
salary                 5
dtype: int64

In [4]:
sal_df[sal_df.age.isna()]

,age,gender,education_level,job_title,years_of_experience,salary
172,NaN,NaN,NaN,NaN,NaN,NaN
260,NaN,NaN,NaN,NaN,NaN,NaN


In [5]:
sal_df = sal_df.dropna(subset=['age'])
sal_df.isna().sum()

age                    0
gender                 0
education_level        1
job_title              0
years_of_experience    1
salary                 3
dtype: int64

In [6]:
sal_df[sal_df.salary.isna()]

,age,gender,education_level,job_title,years_of_experience,salary
3136,31.0,Male,Master's Degree,Full Stack Engineer,8.0,NaN
5247,26.0,Female,Bachelor's Degree,Social M,NaN,NaN
6455,36.0,Male,Bachelor's Degree,Sales Director,6.0,NaN


In [7]:
sal_df = sal_df.dropna(subset=['salary'])
sal_df.isna().sum()

age                    0
gender                 0
education_level        1
job_title              0
years_of_experience    0
salary                 0
dtype: int64

In [8]:
sal_df.dtypes

age                    float64
gender                  object
education_level         object
job_title               object
years_of_experience    float64
salary                 float64
dtype: object

In [9]:
sal_df.gender.unique()

array(['Male', 'Female', 'Other'], dtype=object)

In [10]:
sal_df.education_level.unique()

array(["Bachelor's", "Master's", 'PhD', "Bachelor's Degree",
       "Master's Degree", nan, 'High School', 'phD'], dtype=object)

In [11]:
sal_df.education_level = sal_df.education_level.fillna('na')
sal_df.education_level = sal_df.education_level.str.lower()
sal_df.education_level = sal_df.education_level.str.split(' ', n=1).str[0]
sal_df.education_level = sal_df.education_level.str.replace("'", "")
sal_df.education_level.unique()

array(['bachelors', 'masters', 'phd', 'na', 'high'], dtype=object)

In [12]:
sal_df.job_title.unique()

array(['Software Engineer', 'Data Analyst', 'Senior Manager',
       'Sales Associate', 'Director', 'Marketing Analyst',
       'Product Manager', 'Sales Manager', 'Marketing Coordinator',
       'Senior Scientist', 'Software Developer', 'HR Manager',
       'Financial Analyst', 'Project Manager', 'Customer Service Rep',
       'Operations Manager', 'Marketing Manager', 'Senior Engineer',
       'Data Entry Clerk', 'Sales Director', 'Business Analyst',
       'VP of Operations', 'IT Support', 'Recruiter', 'Financial Manager',
       'Social Media Specialist', 'Software Manager', 'Junior Developer',
       'Senior Consultant', 'Product Designer', 'CEO', 'Accountant',
       'Data Scientist', 'Marketing Specialist', 'Technical Writer',
       'HR Generalist', 'Project Engineer', 'Customer Success Rep',
       'Sales Executive', 'UX Designer', 'Operations Director',
       'Network Engineer', 'Administrative Assistant',
       'Strategy Consultant', 'Copywriter', 'Account Manager',
      

In [13]:
sal_df.isna().sum()

age                    0
gender                 0
education_level        0
job_title              0
years_of_experience    0
salary                 0
dtype: int64

In [14]:
full_train_df, test_df = train_test_split(sal_df, test_size=0.2, random_state=55)
train_df, val_df = train_test_split(full_train_df, test_size=0.25, random_state=55)
(len(train_df), len(val_df), len(test_df))

(4019, 1340, 1340)

In [15]:
full_train_y = full_train_df.salary.values
test_y = test_df.salary.values
train_y = train_df.salary.values
val_y = val_df.salary.values

del full_train_df['salary']
del test_df['salary']
del train_df['salary']
del val_df['salary']

## Training models

### Training a Linear Regression model

In [16]:
def dict_and_vectorize(tr_df, v_df):
    dv = DictVectorizer(sparse=False)
    train_dict = tr_df.to_dict(orient='records')
    val_dict = v_df.to_dict(orient='records')

    train_X = dv.fit_transform(train_dict)
    val_X = dv.transform(val_dict)

    return train_X, val_X, dv

In [17]:
def train_lin_reg(tr_df, tr_y, v_df, v_y, r):
    train_X, val_X, dv = dict_and_vectorize(tr_df, v_df)

    model = Ridge(alpha=r, random_state=55)
    model.fit(train_X, tr_y)

    pred_y = model.predict(val_X)

    return root_mean_squared_error(val_y, pred_y)

In [18]:
linreg_result = []
for r in range(1, 10000, 5):
    r_val = r / 100
    rmse = train_lin_reg(train_df, train_y, val_df, val_y, r_val)
    linreg_result.append((r_val, rmse))

linreg_result_df = pd.DataFrame(
    linreg_result,
    columns=['r', 'rmse']
)
linreg_result_df.sort_values(by='rmse', ascending=True).head()

,r,rmse
10,0.51,21273.382888
11,0.56,21273.461541
9,0.46,21273.551702
12,0.61,21273.755578
8,0.41,21274.006316


### Training a Decision Tree model

In [19]:
def train_dtree(tr_df, tr_y, val_df, val_y, depth):
    train_X, val_X, dv = dict_and_vectorize(tr_df, val_df)

    dt = DecisionTreeRegressor(max_depth=depth, random_state=55)
    dt.fit(train_X, train_y)

    pred_y = dt.predict(val_X)

    return root_mean_squared_error(val_y, pred_y)

In [20]:
dtree_result = []
for depth in range(1, 30, 5):
    rmse = train_dtree(train_df, train_y, val_df, val_y, depth)
    dtree_result.append((depth, rmse))

dtree_result_df = pd.DataFrame(
    dtree_result,
    columns=['depth', 'rmse']
)
dtree_result_df.sort_values(by='rmse', ascending=True).head()

,depth,rmse
4,21,7848.913748
5,26,7984.450207
3,16,8559.699008
2,11,10068.943690
1,6,16268.165439


### Training a Random Forest model

In [21]:
def train_rforest(tr_df, tr_y, val_df, val_y, n_e, m_d):
    train_X, val_X, dv = dict_and_vectorize(tr_df, val_df)

    rf = RandomForestRegressor(n_estimators=n_e, max_depth=m_d, random_state=55)
    rf.fit(train_X, train_y)

    pred_y = rf.predict(val_X)

    return root_mean_squared_error(val_y, pred_y)

In [22]:
rforest_result = []
for depth in range(1, 30, 5):
    for n_estimators in range(10, 201, 10):
        rmse = train_rforest(
            train_df,
            train_y,
            val_df,
            val_y,
            n_estimators,
            depth
        )
        rforest_result.append((n_estimators, depth, rmse))

rforest_result_df = pd.DataFrame(
    rforest_result,
    columns=['n_estimator', 'depth', 'rmse']
)
rforest_result_df.sort_values(by='rmse', ascending=True).head()

,n_estimator,depth,rmse
103,40,26,7070.299272
102,30,26,7084.699236
83,40,21,7116.481397
82,30,21,7135.261231
104,50,26,7152.173344


### Training XGBoost model

In [23]:
def train_xgboost(tr_df, tr_y, val_df, val_y, md, nbr, mce, eta):
    train_X, val_X, dv = dict_and_vectorize(tr_df, val_df)
    feature_names = list(dv.get_feature_names_out())
    train_dm = xgb.DMatrix(train_X, label=tr_y, feature_names=feature_names)
    val_dm = xgb.DMatrix(val_X, label=val_y, feature_names=feature_names)
    watchlist = [(train_dm, 'train'), (val_dm, 'val')]
    
    eval_result = {}
    xgb_params = {
        'eta': eta,
        'max_depth': md,
        'min_child_weight': mce,

        'objective': 'reg:squarederror',
        'nthread': 4,

        'seed': 55,
        'verbosity': 0
    }

    model = xgb.train(
        xgb_params,
        train_dm,
        num_boost_round=nbr,
        evals=watchlist,
        evals_result=eval_result
    )
    return eval_result['val']['rmse']
        

In [24]:
%%capture output
## suppressing logs

xgboost_result = []
## changing to a much smaller range based on original findings
# to make future end-to-end computes much more efficient
#for depth in range(1, 30, 5):
#    for min_child_weight in range(1, 11, 1):
#        for eta in range(1, 10, 1):
for depth in range(6, 16, 5):
    for min_child_weight in range(1, 4, 1):
        for eta in range(4, 8, 1):
            eta_val = eta / 10
            rmse = train_xgboost(
                train_df,
                train_y,
                val_df,
                val_y,
                depth,
                200,
                min_child_weight,
                eta_val
            )
            min_rmse_boost_round = pd.Series(rmse).idxmin() + 1
            rmse_val = min(rmse)
            xgboost_result.append((
                depth,
                min_rmse_boost_round,
                min_child_weight,
                eta_val,
                rmse_val
            ))

xgboost_result_df = pd.DataFrame(
    xgboost_result,
    columns=['depth', 'num_boost_round', 'min_child_weight', 'eta', 'rmse']
)

In [25]:
xgboost_result_df.sort_values(by='rmse', ascending=True).head()

,depth,num_boost_round,min_child_weight,eta,rmse
17,11,179,2,0.5,7202.081497
16,11,81,2,0.4,7225.328645
12,11,136,1,0.4,7270.365294
11,6,200,3,0.7,7501.665007
13,11,67,1,0.5,7513.935621


## Selecting model

In [26]:
print('Best Linear Regression:')
display(linreg_result_df.sort_values(by='rmse', ascending=True).head(1))
print('Best Decision Tree:')
display(dtree_result_df.sort_values(by='rmse', ascending=True).head(1))
print('Best Random Forest:')
display(rforest_result_df.sort_values(by='rmse', ascending=True).head(1))
print('Best XGBoost:')
display(xgboost_result_df.sort_values(by='rmse', ascending=True).head(1))

Best Linear Regression:


,r,rmse
10,0.51,21273.382888


Best Decision Tree:


,depth,rmse
4,21,7848.913748


Best Random Forest:


,n_estimator,depth,rmse
103,40,26,7070.299272


Best XGBoost:


,depth,num_boost_round,min_child_weight,eta,rmse
17,11,179,2,0.5,7202.081497


The best `rmse` comes from the Random Forest model.

## Setup Deployment Environment

In [27]:
# I've installed uv using the requirements.txt file
# in the docker container hosting my jupyter notebook
!uv --version

uv 0.9.5


In [28]:
# I will only run this once, since the lockfile will be committed
#!uv init
#!uv add pandas==2.3.2 scikit-learn==1.7.2 pickle-mixin==1.0.2 fastapi==0.121.1 uvicorn==0.38.0

## Testing Deployed Service
This is here for demonstration simplicity.  The service and model were defined ahead of time and built when you stood up the docker containers using: 
```sh
docker compose up -d midterm
```
Changes to any model definitions in this notebook will _not_ impact the running service.  You will have to change `train.py` and `main.py` to change the running service.

In [29]:
url = 'http://midterm-service:9696/predict'

# values for testing convenience
genders = list(sal_df.gender.unique())
education_levels = list(sal_df.education_level.unique())
job_titles = list(sal_df.job_title.unique())

employee = {
    "age": 28,
    "gender": genders[0],
    "education_level": education_levels[0],
    "job_title": job_titles[55],
    "years_of_experience": 7.0
}

print(employee)

response = requests.post(url, json=employee)
response.json()

{'age': 28, 'gender': 'Male', 'education_level': 'bachelors', 'job_title': 'Director of Operations', 'years_of_experience': 7.0}


{'predicted_salary': 120351.35}